# Projet MD5 — HTR + NLP sur Google Colab (version propre)

Notebook complet et robuste : exécuter les cellules **dans l'ordre, sans en sauter**.

**Avant de commencer :** activer le GPU.
→ Menu `Exécution` > `Modifier le type d'exécution` > **T4 GPU** > Enregistrer.

⚠️ **Règle d'or** : les installations (étape 1) se font AVANT tout import — ainsi, pas besoin de redémarrer la session.

## 1. Installer les dépendances (AVANT tout import)

On fige `transformers` sur une version stable et on installe uniquement ce qui manque.
Les avertissements pip en rouge (gradio, datasets, jax...) sont **bénins** : ignorez-les.

In [ ]:
!pip install -q "transformers==4.46.3"
!pip install -q peft editdistance jiwer jsonschema stanza networkx
!pip uninstall -y -q mlflow torchao 2>/dev/null
print("Installation terminee.")

## 2. Vérifier le GPU et les versions

In [ ]:
import torch, transformers
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Modele GPU     :', torch.cuda.get_device_name(0))
print('transformers   :', transformers.__version__)  # doit afficher 4.46.3

## 3. Charger le projet

**Choisir UNE option** : A (GitHub) ou B (upload du ZIP). Puis exécuter la cellule de normalisation (elle gère le dossier imbriqué et le ZIP-dans-le-dépôt).

In [ ]:
# ── OPTION A : depuis GitHub ─────────────────────────────────────
%cd /content
!rm -rf htr-medieval-manuscripts-2026
!git clone https://github.com/TNAJ1/htr-medieval-manuscripts-2026.git

In [ ]:
# ── OPTION B : upload du ZIP (si le depot GitHub est vide/incomplet) ──
from google.colab import files
import zipfile, os
%cd /content
up = files.upload()
nom = list(up.keys())[0]
with zipfile.ZipFile(nom) as z:
    z.extractall('/content')
print("Extrait. Dossiers presents :", [d for d in os.listdir('/content') if 'htr' in d])

In [ ]:
# ── NORMALISATION du chemin (OBLIGATOIRE apres A ou B) ──
import os, sys, glob, shutil

RACINE = '/content/htr-medieval-manuscripts-2026'

# Cas 1 : le depot ne contient qu'un ZIP -> l'extraire sur place
if os.path.exists(RACINE) and not os.path.exists(os.path.join(RACINE, 'src')):
    zips = glob.glob(os.path.join(RACINE, '*.zip'))
    if zips:
        import zipfile
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(RACINE)

# Cas 2 : dossier imbrique -> remonter le contenu
imbrique = os.path.join(RACINE, 'htr-medieval-manuscripts-2026')
if os.path.exists(os.path.join(imbrique, 'src')):
    for item in os.listdir(imbrique):
        dest = os.path.join(RACINE, item)
        if not os.path.exists(dest):
            shutil.move(os.path.join(imbrique, item), dest)
    shutil.rmtree(imbrique, ignore_errors=True)

os.chdir(RACINE)
sys.path.insert(0, os.path.join(RACINE, 'src'))

assert os.path.exists('src/htr/finetuning.py'), "src/htr/finetuning.py introuvable !"
print("OK — projet en place :", os.getcwd())

## 4. Vérifier que les correctifs sont présents

Chaque compteur doit afficher **au moins 1** (0 = code trop ancien ; 2+ = patch en double — à signaler).

In [ ]:
!grep -c 'report_to' src/htr/finetuning.py
!grep -c 'generation_max_length' src/htr/finetuning.py
!grep -c 'np.where' src/htr/finetuning.py
!grep -c 'generation_config' src/htr/finetuning.py

## 5. Télécharger le corpus CREMMA (~2 min)

In [ ]:
import os
if not os.path.exists('data/raw/cremma-medieval'):
    !git clone https://github.com/HTR-United/cremma-medieval data/raw/cremma-medieval
else:
    print("Corpus deja present.")

## 6. Charger le corpus et le découper (train/val/test)

In [ ]:
from htr.corpus_loader import charger_corpus
from htr.dataset_split import split_stratifie

exemples = charger_corpus('data/raw/cremma-medieval',
                          metadata={'century': 13, 'source': 'CREMMA'})
splits = split_stratifie(exemples)
print(f"train={len(splits['train'])}  val={len(splits['val'])}  test={len(splits['test'])}")

## 7. Fine-tuning TrOCR — version COURTE (~1h-1h30)

Sous-échantillon (6000 lignes) + 3 epochs : suffisant pour une courbe qui descend et un CER de validation.

⚠️ Garder l'onglet Colab actif pendant l'entraînement.

À la fin : **noter le « Meilleur CER de validation »** — c'est le chiffre pour la slide 11.

In [ ]:
from htr.finetuning import finetuner_trocr, ConfigEntrainement

historique = finetuner_trocr(
    splits['train'][:6000],
    splits['val'][:800],
    config=ConfigEntrainement(epochs=3, lora_r=8),
    mode_simulation=False,
)
print('Meilleur CER de validation :', historique['meilleur_cer'])

## 8. SAUVEGARDER sur Google Drive (tout de suite après l'entraînement !)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/md5_resultats
!cp -r models experiments /content/drive/MyDrive/md5_resultats/ 2>/dev/null
print("Sauvegarde sur Drive.")

## 9. Générer les manuscrits colorés (pour la slide 6)

Segmentation colorée + transcriptions en regard, sur de vraies pages CREMMA.
Télécharger ensuite les PNG (panneau Fichiers à gauche > clic droit > Télécharger).

In [ ]:
from htr.visualisation import visualiser_page
from htr.corpus_loader import trouver_paires

paires = trouver_paires('data/raw/cremma-medieval')
print(f"{len(paires)} pages disponibles")

for i in [0, 50, 100]:
    img, xml = paires[i]
    visualiser_page(img, xml, f'resultats/manuscrit_{i}.png')

from IPython.display import Image, display
display(Image('resultats/manuscrit_0.png', width=900))

## 10. (Option) Évaluation finale — CER sur données jamais vues

⚠️ Long si on transcrit tout. `limite=500` donne un premier chiffre rapide.

In [ ]:
from htr.evaluation_corpus import evaluer_modele_sur_corpus

rapport = evaluer_modele_sur_corpus(
    'data/raw/cremma-medieval',
    modele='trocr',
    metadata={'century': 13, 'source': 'CREMMA'},
    mode_simulation=False,
    limite=500,
)
print(f"CER : {rapport['CER']:.2%}  |  WER : {rapport['WER']:.2%}")

## 11. Pousser la VRAIE structure sur GitHub

Le dépôt doit contenir le code décompressé (src/, tests/, docs), **pas un ZIP**.
Remplacer `<TON_TOKEN>` par un token GitHub (Settings > Developer settings > Personal access tokens > Tokens classic, cocher `repo`). Ne jamais partager ce token.

In [ ]:
%cd /content/htr-medieval-manuscripts-2026
# Ne pas versionner : le corpus, les ZIP, les caches, les modeles lourds
!rm -rf data/raw/cremma-medieval .pytest_cache
!rm -f *.zip
!find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null

!git init -q 2>/dev/null
!git config user.email 'ton@email.com'
!git config user.name 'TNAJ1'
!git add -A
!git commit -q -m 'Projet MD5 - structure complete (code, tests, docs)'
!git branch -M main
!git remote remove origin 2>/dev/null
!git remote add origin https://<TON_TOKEN>@github.com/TNAJ1/htr-medieval-manuscripts-2026.git
!git push -f -u origin main
print('Depot GitHub mis a jour.')